In [ ]:
#THIS NOTEBOOK IS THE BASELINE PREDICTION MODEL FOR MELTING TEMPERATURE (Tm)

from chemprop_utils import load_and_clean_data, build_splits, build_and_train_mpnn, predict_test_set, predict_on_set    #IMPORTS REQUIRED FUNCTIONS FROM THE FUNCTION SCRIPT
from pathlib import Path                    #IMPORTS THE FILE PATH HANDLING LIBRARY
import pandas as pd                         #IMPORTS THE DATA FRAME HANDLING LIBRARY
from lightning import pytorch as pl         #IMPORTS PYTORCH LIGHTNING, USED FOR SEEDING AND TRAINING

pl.seed_everything(0, workers=True)         #FIXES THE RANDOM SEED FOR REPRODUCIBILITY (ALSO SEEDS DATALOADER WORKER PROCESSES)

# SPLIT: SCAFFOLD_BALANCED SPLIT WILL BE USED AS IT OUTPERFORMED RANDOM SPLIT RUNS
# RMSE WILL BE USED AS TRAINING OBJECTIVE — SAME REASONING AS dHfus BASELINE
# DROPOUT OF 0.4 WILL BE USED FOR CONSISTENCY WITH dHfus BASELINE (NOT INDEPENDENTLY RE-TUNED FOR Tm)
# NO FOLDING WILL BE DONE (SINGLE 80/10/10 SPLIT) TO INCREASE RUN SPEED


input_path = Path("RAW DATA") / "Tmp_DB.csv"
smiles_column = "smiles"                                 #DEFINES THE SMILES COLUMN
target_columns = ["Tm[K]"]                               #DEFINES THE MELTING TEMPERATURE COLUMN AS THE TARGET

print(pd.read_csv(input_path).columns.tolist())

import pandas as pd
df = pd.read_csv(input_path)
print("NaN SMILES:", df["smiles"].isna().sum())
print(df[df["smiles"].isna()])


df_clean, all_data, mols = load_and_clean_data(input_path, smiles_column, target_columns)
splits = build_splits(all_data, mols, split_types=("scaffold_balanced",))

result_baseline = build_and_train_mpnn(
    splits["scaffold_balanced"]["train"],
    splits["scaffold_balanced"]["val"],
    splits["scaffold_balanced"]["test"],
    dropout=0.4,
    run_name="Tm_OFFICIAL_BASELINE",
)

test_metrics = result_baseline["test_results"][0]
print(f"OFFICIAL Tm BASELINE — RMSE: {test_metrics['test/rmse']:.3f}, MAE: {test_metrics['test/mae']:.3f}")

baseline_predictions = predict_test_set(result_baseline, run_name="Tm_OFFICIAL_BASELINE")
baseline_predictions.to_csv("Tm_baseline_predictions.csv", index=False)

#SAVE TRAINING LOSS CURVE DATA AND TRAIN-SET PREDICTIONS FOR USE IN THE Tm RESULTS NOTEBOOK
log_dir = result_baseline["trainer"].logger.log_dir
metrics_df = pd.read_csv(f"{log_dir}/metrics.csv")
metrics_df.to_csv("Tm_baseline_loss_curve.csv", index=False)

train_predictions = predict_on_set(result_baseline, "train_dset", "train_loader", run_name="Tm_train")
train_predictions.to_csv("Tm_baseline_train_predictions.csv", index=False)

#NOTE: RESULTS AND SPLITS MUST BE SAVED TO BE USED IN SUBSEQUENT NOTEBOOKS (E.G. MULTI-TASK)
import pickle
with open("Tm_baseline_splits.pkl", "wb") as f:
    pickle.dump(splits, f)

print("Tm baseline locked in. Predictions, splits, loss curve, and train predictions saved for downstream notebooks.")

Seed set to 0


['inchi_fixed', 'inchi', 'smiles', 'Tm[K]', 'data_source', 'source', 'heavy_atom_count', 'molar_mass', 'tag', 'Tm_reduced', 'count', 'avg', 'std', 'molar_mass_log', 'use']
NaN SMILES: 0
Empty DataFrame
Columns: [inchi_fixed, inchi, smiles, Tm[K], data_source, source, heavy_atom_count, molar_mass, tag, Tm_reduced, count, avg, std, molar_mass_log, use]
Index: []


[11:52:54] Explicit valence for atom # 2 S, 6, is greater than permitted
[11:52:56] Explicit valence for atom # 2 S, 6, is greater than permitted
[11:52:59] Explicit valence for atom # 8 S, 7, is greater than permitted
[11:53:00] WARNING: not removing hydrogen atom without neighbors
[11:53:11] Explicit valence for atom # 1 S, 6, is greater than permitted
[11:53:11] Explicit valence for atom # 13 S, 6, is greater than permitted
[11:53:11] Explicit valence for atom # 2 S, 6, is greater than permitted


Dropping 6 rows with unparseable SMILES
Loaded 273236 rows, dropped 6 with missing target, 273230 remain


[11:53:36] WARNING: not removing hydrogen atom without neighbors
The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
[11:54:58] 

****
Pre-condition Violation
bad bond stereo
Violation occurred on line 362 in file C:\rdkit\build\temp.win-amd64-cpython-314\Release\rdkit\Code\GraphMol\Canon.cpp
Failed Expression: dblBond->getStereo() > Bond::STEREOANY
****



RuntimeError: Pre-condition Violation
	bad bond stereo
	Violation occurred on line 362 in file Code\GraphMol\Canon.cpp
	Failed Expression: dblBond->getStereo() > Bond::STEREOANY
	RDKIT: 2026.03.1
	BOOST: 1_85
